In [1]:
import numpy as np
from PIL import Image # này là thư viện xử lý ảnh nó sẽ mở những ảnh có đuôi (.jpg, .png)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [2]:
# đây là bước chuẩn hóa dữ liệu đầu vào cho model
# vì dữ liệu ảnh nó nằm trong khoảng pixel từ 0 - 255
transform = transforms.Compose([
    transforms.ToTensor(), # dòng này sẽ biến đổi ảnh thành 1 tensor và chuẩn hóa nó để nằm trong khoảng từ 0 - 1 (có nghĩa là 0 - 255 sẽ được thu nhỏ xuống còn 0-1)
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # dòng này giúp chuẩn hóa từ khoảng của tensor là 0-1 thành -1 - 1
    # các số 0.5 ở trước dành cho mỗi kênh, 0.5 ở sau dành cho độ lệch chuẩn cho mỗi kênh
])

In [3]:
# đây là phần lấy dữ liệu để huấn luyện 
train_data = torchvision.datasets.CIFAR10(root="./data", train=True, transform=transform, download=True)
test_data = torchvision.datasets.CIFAR10(root="./data", train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=True, num_workers=2)

c:\Users\Danh\Desktop\AI\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
# đây là phần kiểm tra đầu vào trước khi train
image, label = train_data[0]

In [5]:
image.size()

torch.Size([3, 32, 32])

In [6]:
# đây là những output của model 
# và là nhãn cho 10 lớp vì mạng nơ ron sẽ tạo ra một số từ 1 - 9 đại diện cho từng lớp riêng lẻ 
class_names = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

In [ ]:
# đây là phần tạo ra mô hình mạng nơ ron kế thừa từ nn.Module cái này nó sẽ đi vào GPU
# cái này làm cái deoo gì và toán nó sẽ sử dụng cho cái gì 
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 12, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(12, 24, 5)
        self.fc1 = nn.Linear(24 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [8]:
net = NeuralNet()
loss_function = nn.CrossEntropyLoss() # cái này là tạo ra hàm mất mát nó dùng để do xem model sai bao nhiêu 
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9) # cái này là giúp định nghĩa các model học
#momentum là gi?

In [9]:
# đây là vòng lặp huấn luyện nó giúp cho model học từ dữ liệu và giảm sai số, sai số càng thấp thì model càng tốt
for epoch in range(30):
    print(f"Training epoch {epoch}...")

    running_loss = 0.0

    for i, data in enumerate(train_loader):
        inputs, labels = data

        optimizer.zero_grad()

        outputs = net(inputs)

        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Loss: {running_loss / len(train_loader): .4f}")

Training epoch 0...
Loss:  2.2653
Training epoch 1...
Loss:  1.8089
Training epoch 2...
Loss:  1.5603
Training epoch 3...
Loss:  1.4209
Training epoch 4...
Loss:  1.3206
Training epoch 5...
Loss:  1.2430
Training epoch 6...
Loss:  1.1648
Training epoch 7...
Loss:  1.1038
Training epoch 8...
Loss:  1.0414
Training epoch 9...
Loss:  0.9949
Training epoch 10...
Loss:  0.9489
Training epoch 11...
Loss:  0.9083
Training epoch 12...
Loss:  0.8733
Training epoch 13...
Loss:  0.8337
Training epoch 14...
Loss:  0.8001
Training epoch 15...
Loss:  0.7704
Training epoch 16...
Loss:  0.7398
Training epoch 17...
Loss:  0.7112
Training epoch 18...
Loss:  0.6850
Training epoch 19...
Loss:  0.6595
Training epoch 20...
Loss:  0.6342
Training epoch 21...
Loss:  0.6123
Training epoch 22...
Loss:  0.5884
Training epoch 23...
Loss:  0.5635
Training epoch 24...
Loss:  0.5432
Training epoch 25...
Loss:  0.5222
Training epoch 26...
Loss:  0.5006
Training epoch 27...
Loss:  0.4818
Training epoch 28...
Loss:  0.

In [10]:
# cái này nó lưu lại các cái trọng số vào 1 file 
torch.save(net.state_dict(), "trained_net.pth")
# pth là gì?

In [11]:
# cái này là mình sẽ load lại mỗi khi cần
net = NeuralNet()
net.load_state_dict(torch.load("trained_net.pth"))

<All keys matched successfully>

In [12]:
# đây là bước đánh giá xem model có thực sự tốt hay không 
correct = 0
total = 0

net.eval()
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy}%")

Accuracy: 69.77%


In [ ]:
new_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def load_image(image_path):
    image = Image.open(image_path)
    image = new_transform(image)
    image = image.unsqueeze(0)
    return image

image_paths = ['concat.jpg', 'condog.jpg', "image.png"]
images = [load_image(img) for img in image_paths]

net.eval()
with torch.no_grad():
    for image in images:
        output = net(image)
        _, predicted = torch.max(output, 1)
        print(f"Prediction: {class_names[predicted.item()]}")


# coi cái đây cho kĩ tự xem cái gì hay ho thì teamwork gửi cho nhau

Prediction: cat
Prediction: dog
Prediction: cat
